# Module 2 — Arrays and Dynamic Arrays

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a1-dynamic-array/starter/dynamic_array.py`.

## 1. Address arithmetic, made concrete

Lecture 1's formula: `address(i) = base + i * element_size`. This is not
just theory — here it is as a two-line function, checked against the
worked examples from lecture.

In [1]:
def address(base, i, element_size):
    return base + i * element_size

assert address(500, 0, 4) == 500
assert address(500, 1, 4) == 504
assert address(500, 5, 4) == 520
assert address(3000, 7, 4) == 3028
print("1D address arithmetic checks passed")

1D address arithmetic checks passed


In [2]:
def address_2d(base, row, col, num_cols, element_size):
    return base + (row * num_cols + col) * element_size

assert address_2d(8000, 2, 1, 3, 4) == 8028
assert address_2d(0, 3, 2, 4, 4) == 56
print("2D row-major address arithmetic checks passed")

2D row-major address arithmetic checks passed


## 2. `DynamicArray` from scratch

This is the fully-solved version of the Lab 2, Part B exercise. It backs
itself with a fixed-size Python list used purely as "raw memory" — every
resize allocates a brand new backing list, copies elements across, and
discards the old one, exactly as traced in Lecture 2.

In [3]:
class DynamicArray:
    def __init__(self):
        self.capacity = 1
        self.length = 0
        self._backing = [None] * self.capacity

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        if i >= self.length or i < 0:
            raise IndexError(f"index {i} out of range for length {self.length}")
        return self._backing[i]

    def _resize(self, new_capacity):
        new_backing = [None] * new_capacity
        for i in range(self.length):
            new_backing[i] = self._backing[i]
        print(f"RESIZE: capacity {self.capacity} -> {new_capacity}")
        self._backing = new_backing
        self.capacity = new_capacity

    def append(self, x):
        if self.length == self.capacity:
            self._resize(self.capacity * 2)
        self._backing[self.length] = x
        self.length += 1

In [4]:
arr = DynamicArray()
for i in range(20):
    arr.append(i)

assert len(arr) == 20
assert arr[0] == 0 and arr[19] == 19
assert arr.capacity == 32   # doubling sequence: 1,2,4,8,16,32
print("DynamicArray append + resize checks passed, final capacity:", arr.capacity)

RESIZE: capacity 1 -> 2
RESIZE: capacity 2 -> 4
RESIZE: capacity 4 -> 8
RESIZE: capacity 8 -> 16
RESIZE: capacity 16 -> 32
DynamicArray append + resize checks passed, final capacity: 32


## 3. Confirming the doubling sequence matches Lecture 2's trace

Resizes should occur exactly when length first exceeds capacity 1, 2, 4, 8,
16 — that is, at append numbers 2, 3, 5, 9, 17.

In [5]:
arr2 = DynamicArray()
capacities = []
for i in range(20):
    arr2.append(i)
    capacities.append(arr2.capacity)

expected_resizes_at = [2, 3, 5, 9, 17]
actual_resizes_at = [n for n in range(1, 21) if capacities[n-1] != (capacities[n-2] if n > 1 else 1)]
print("capacity after each append:", capacities)
assert actual_resizes_at == expected_resizes_at, actual_resizes_at
print("Resize points match Lecture 2's hand-traced doubling sequence")

RESIZE: capacity 1 -> 2
RESIZE: capacity 2 -> 4
RESIZE: capacity 4 -> 8
RESIZE: capacity 8 -> 16
RESIZE: capacity 16 -> 32
capacity after each append: [1, 2, 4, 4, 8, 8, 8, 8, 16, 16, 16, 16, 16, 16, 16, 16, 32, 32, 32, 32]
Resize points match Lecture 2's hand-traced doubling sequence


## 4. Measuring append vs. front-insert (Lecture 4, made real)

This reproduces the timing comparison from Lecture 4: appending at the end
of a plain Python `list` versus inserting at index 0. The gap should widen
as `n` grows, evidence of O(1) versus O(n).

In [6]:
import time

def time_append(n):
    data = []
    start = time.perf_counter()
    for i in range(n):
        data.append(i)
    return time.perf_counter() - start

def time_front_insert(n):
    data = []
    start = time.perf_counter()
    for i in range(n):
        data.insert(0, i)
    return time.perf_counter() - start

for n in (2_000, 4_000, 8_000):
    t_append = time_append(n)
    t_front = time_front_insert(n)
    print(f"n={n:>6}  append={t_append:.4f}s  insert(0)={t_front:.4f}s  ratio={t_front/t_append:.1f}x")

n=  2000  append=0.0001s  insert(0)=0.0008s  ratio=6.7x
n=  4000  append=0.0002s  insert(0)=0.0032s  ratio=15.3x
n=  8000  append=0.0004s  insert(0)=0.0127s  ratio=29.7x


Notice the ratio column growing as `n` doubles — `time_append` stays close
to linear in `n` (amortized O(1) per call), while `time_front_insert` grows
closer to quadratic in `n` (O(n) per call, from Lecture 4's shifting
argument), so the *total* cost over `n` front-inserts is O(n²).

## 5. Sorted insertion: binary search finds the spot, shifting still costs O(n)

This reproduces the full worked trace from the end of Lecture 4: use binary
search to find the correct position in a sorted list, then insert there.

In [7]:
def find_sorted_position(arr, x):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

def insert_sorted(arr, x):
    pos = find_sorted_position(arr, x)
    arr.insert(pos, x)
    return arr

leaderboard = [45, 60, 71, 88, 92, 99]
insert_sorted(leaderboard, 80)
assert leaderboard == [45, 60, 71, 80, 88, 92, 99]
print("Sorted insertion matches Lecture 4's worked trace:", leaderboard)

Sorted insertion matches Lecture 4's worked trace: [45, 60, 71, 80, 88, 92, 99]
